### Hyperparameter Tuning With GridSearch:
* It involves thoroughly searching a specified parameter grid to identify the best hyperparameter combination for machine learning model. This is done by evaluating performance through cross-validation.

In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier

X,y = make_classification(
    n_classes=2,
    n_informative=8,
    n_redundant=2,
    n_repeated=0,
    n_samples=1000,
    n_features=10,
    random_state=42,
    )

### Method 1: Evaluate the model using train test split and tune parameteres by trial and error

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = DecisionTreeClassifier(criterion="entropy", max_depth=10) # criteria: gini or entropy, max_depth: 5 or 10
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

report = classification_report(y_test, y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.84      0.75      0.79       130
           1       0.75      0.84      0.80       120

    accuracy                           0.79       250
   macro avg       0.79      0.79      0.79       250
weighted avg       0.80      0.79      0.79       250



But in this approach we have to calculate different parameters (i.e criteria, max_depth, etc.) or for differnt classifiers manually which is time consuming and tedious process. 

### Method 2: Using k-fold cross validation score.

In [13]:
from sklearn.model_selection import cross_val_score

cross_val_score(DecisionTreeClassifier(criterion="gini", max_depth=5), X, y, cv = 5)

array([0.775, 0.79 , 0.74 , 0.795, 0.775])

In [14]:
cross_val_score(DecisionTreeClassifier(criterion="entropy", max_depth=5), X, y, cv = 5)

array([0.765, 0.785, 0.76 , 0.81 , 0.78 ])

In [15]:
cross_val_score(DecisionTreeClassifier(criterion="gini", max_depth=10), X, y, cv = 5)

array([0.77 , 0.72 , 0.785, 0.785, 0.8  ])

In [16]:
cross_val_score(DecisionTreeClassifier(criterion="entropy", max_depth=10), X, y, cv = 5)

array([0.77 , 0.79 , 0.84 , 0.78 , 0.785])

Let's make it little faster and convinient using for loops

In [19]:
criterion = ['gini', 'entropy']
max_depth = [5, 10, 15]

avg_scores = {}

for c in criterion:
    for d in max_depth:
        clf = DecisionTreeClassifier(criterion=c, max_depth=d)
        score_list = cross_val_score(clf, X, y, cv = 5)
        avg_scores[c+"_"+str(d)] = np.average(score_list)
avg_scores

{'gini_5': 0.7849999999999999,
 'gini_10': 0.785,
 'gini_15': 0.7969999999999999,
 'entropy_5': 0.78,
 'entropy_10': 0.7940000000000002,
 'entropy_15': 0.812}

Luckily scikit learn provides a convient API to do all of this whihc is GridSearchCV

### Method 3: Using GridSearchCV (Efficient way)

In [20]:
from sklearn.model_selection import GridSearchCV

clf = GridSearchCV(  # Creating obj
    DecisionTreeClassifier(),
    {
        'criterion' : ['gini', 'entropy'],
        'max_depth' : [5, 10, 15]
    }, 
    cv = 5, 
    return_train_score= False
)

# let's now fit it on our data
clf.fit(X, y) # No need to do train_test split coz cv internally create folds

,estimator,DecisionTreeClassifier()
,param_grid,"{'criterion': ['gini', 'entropy'], 'max_depth': [5, 10, ...]}"
,scoring,None
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'entropy'


In [22]:
clf.cv_results_ # Let's create DataFrame of it so we can easily see it

{'mean_fit_time': array([0.00878286, 0.01078167, 0.0105792 , 0.0113028 , 0.01626425,
        0.01813397]),
 'std_fit_time': array([0.00147918, 0.00126594, 0.00168473, 0.00064749, 0.00093387,
        0.00108096]),
 'mean_score_time': array([0.0014257 , 0.00103192, 0.00060015, 0.00100079, 0.00099931,
        0.00100212]),
 'std_score_time': array([4.81357544e-04, 5.38436390e-04, 4.90018183e-04, 6.33390223e-04,
        1.13443158e-06, 3.44314732e-06]),
 'param_criterion': masked_array(data=['gini', 'gini', 'gini', 'entropy', 'entropy',
                    'entropy'],
              mask=[False, False, False, False, False, False],
        fill_value='?',
             dtype=object),
 'param_max_depth': masked_array(data=[5, 10, 15, 5, 10, 15],
              mask=[False, False, False, False, False, False],
        fill_value=999999),
 'params': [{'criterion': 'gini', 'max_depth': 5},
  {'criterion': 'gini', 'max_depth': 10},
  {'criterion': 'gini', 'max_depth': 15},
  {'criterion': 'entropy',

In [25]:
df = pd.DataFrame(clf.cv_results_)
df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_depth,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.008783,0.001479,0.001426,0.000481,gini,5,"{'criterion': 'gini', 'max_depth': 5}",0.780,0.815,0.750,0.795,0.770,0.782,0.022045,5
1,0.010782,0.001266,0.001032,0.000538,gini,10,"{'criterion': 'gini', 'max_depth': 10}",0.795,0.755,0.815,0.790,0.815,0.794,0.022000,3
2,0.010579,0.001685,0.000600,0.000490,gini,15,"{'criterion': 'gini', 'max_depth': 15}",0.800,0.730,0.810,0.830,0.825,0.799,0.036111,2
3,0.011303,0.000647,0.001001,0.000633,entropy,5,"{'criterion': 'entropy', 'max_depth': 5}",0.765,0.785,0.755,0.815,0.780,0.780,0.020494,6
4,0.016264,0.000934,0.000999,0.000001,entropy,10,"{'criterion': 'entropy', 'max_depth': 10}",0.770,0.780,0.835,0.780,0.805,0.794,0.023537,3
5,0.018134,0.001081,0.001002,0.000003,entropy,15,"{'criterion': 'entropy', 'max_depth': 15}",0.755,0.780,0.830,0.800,0.865,0.806,0.038393,1


In [26]:
# let's just print few columns for better viewing
df[['param_criterion','param_max_depth','mean_test_score']]

,param_criterion,param_max_depth,mean_test_score
0,gini,5,0.782
1,gini,10,0.794
2,gini,15,0.799
3,entropy,5,0.780
4,entropy,10,0.794
5,entropy,15,0.806


In [28]:
# Now if we want to see the best parameters among this we can use best_params_
clf.best_params_

{'criterion': 'entropy', 'max_depth': 15}

In [29]:
# And No need to train model again on this params, Coz it also stores the best model itself as best_estimator_
clf.best_estimator_

,criterion,'entropy'
,splitter,'best'
,max_depth,15
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


So we can directely use this model or deploy it at production level.

Now, what if we want to try different models like svm, RandomForest, etc. also with different params (Like 5 different models with 10 different params)?

We can also do that!

In [42]:
from sklearn.svm import SVC
# store the params first in dict
model_params = {
    'descision_tree' :{
        'model' : DecisionTreeClassifier(),
        'params':{
            'criterion': ['gini', 'entropy'],
            'max_depth': [5, 10, 15]
        }
    },
    'svm' : {
        'model' : SVC(gamma='auto'),
        'params':{
            'C' : [1, 10, 20],
            'kernel' : ['rbf', 'linear']
        }
    }
}

# scores list to store results
scores = []

# Now run loop on it to try out all different combinations
for key, val in model_params.items():
    clf = GridSearchCV(val['model'], val['params'], cv = 5, return_train_score=False)
    clf.fit(X, y)
    scores.append({
        'model' : key,
        'best_params' : clf.best_params_,
        'best_score' : clf.best_score_
    })
scores

[{'model': 'descision_tree',
  'best_params': {'criterion': 'entropy', 'max_depth': 15},
  'best_score': 0.8160000000000001},
 {'model': 'svm',
  'best_params': {'C': 1, 'kernel': 'rbf'},
  'best_score': 0.9260000000000002}]

In [43]:
df = pd.DataFrame(scores)
df

,model,best_params,best_score
0,descision_tree,"{'criterion': 'entropy', 'max_depth': 15}",0.816
1,svm,"{'C': 1, 'kernel': 'rbf'}",0.926


### Conclusion of this perticuler grid search:

* Descision tree model with criterion as entropy and max_depth as 15 gives best_score of 0.81
* Also between DT and SVM, SVM is giving better performance of 0.92 score with params C as 1 and kernal as rbf